# Data Exploration
This notebook explores all the clinical and genomic files provided in the `data/` directory. It gives us a sense of what we are working with before we begin building the bioinformatics pipeline.

In [1]:
import os
from pathlib import Path
import pandas as pd

# Let's see what is in the data directory and how large the files are
data_dir = Path('../data')
files = list(data_dir.glob('*.*'))

file_info = []
for f in sorted(files):
    size_gb = f.stat().st_size / (1024**3)
    size_mb = f.stat().st_size / (1024**2)
    
    size_str = f"{size_gb:.2f} GB" if size_gb > 1 else f"{size_mb:.2f} MB"
    file_info.append({'File': f.name, 'Size': size_str})

display(pd.DataFrame(file_info))

,File,Size
0,.cache,0.00 MB
1,Challenge_Clinical_Phenotype_1.docx,0.02 MB
2,WGS_EX2312012_HGWCNDSX7.vcf.gz,300.55 MB
3,WGS_EX2312012_HGWCNDSX7.vcf.gz.tbi,2.23 MB
4,WGS_EX2312012_HGWCNDSX7_S16_L001_R1_001.fastq.gz,9.75 GB
5,WGS_EX2312012_HGWCNDSX7_S16_L001_R2_001.fastq.gz,10.28 GB
6,WGS_EX2312012_HGWCNDSX7_S16_L002_R1_001.fastq.gz,9.66 GB
7,WGS_EX2312012_HGWCNDSX7_S16_L002_R2_001.fastq.gz,10.16 GB
8,WGS_EX2312012_HGWCNDSX7_S16_L003_R1_001.fastq.gz,9.50 GB
9,WGS_EX2312012_HGWCNDSX7_S16_L003_R2_001.fastq.gz,9.89 GB


## Processed Genomic Data (.vcf.gz)
The VCF file contains the **genotype** differences. We will use `cyvcf2` to inspect the header, verify the sample name, and peek at the first 3 variant calls.

### Understanding the Columns & Coordinates
When we load the VCF into a pandas DataFrame, we extract the most critical columns for each variant:

*   **`CHROM` (Chromosome):** Which of the 23 chromosome pairs (or X/Y sex chromosomes) the variant is located on. E.g., `1` means Chromosome 1.
*   **`POS` (Position):** The exact coordinate (base-pair number) on that chromosome where the difference was found. 
    *   **GRCh38 (Genome Reference Consortium Human Build 38):** Humans share about 99.9% of their DNA. To have a standard way to communicate, scientists sequenced several individuals and combined them into a single "canonical" map or blueprint of the human genome. GRCh38 is the 38th major version of this map (released in 2013). When the VCF says `POS=13273`, it specifically means "position 13273 according to the GRCh38 map."
*   **`REF` (Reference Allele):** The DNA letter(s) — A, C, T, or G — that a "standard" human reference genome has at this exact position.
*   **`ALT` (Alternate Allele):** The DNA letter(s) that **this specific child** actually has.
    *   *Insertions/Deletions:* If `REF` is **`AT`** and `ALT` is just **`A`**, the `T` was deleted. If `REF` is **`C`** and `ALT` is **`CGAGA`**, `GAGA` was inserted.
*   **`QUAL` (Quality Score):** A numerical score showing how confident the sequencing machine and algorithm are that this mutation is real and not just a scanning error. A higher number is better.
*   **`FILTER`:** Did this variant pass the algorithm's quality control checks? Ideally, we want variants that say `PASS`.
*   **`GT` (Genotype):** Tells us *how* the child inherited this mutation. In `cyvcf2`, it outputs a number:
    *   **`1` (Heterozygous):** The child inherited the `ALT` mutation from *one* parent, but the normal `REF` letter from the other parent.
    *   **`3` (Homozygous Alternate):** The child inherited the exact same `ALT` mutation from *both* parents. Both copies of their gene have the same typo.


In [3]:
from cyvcf2 import VCF
import pandas as pd

vcf_path = '../data/WGS_EX2312012_HGWCNDSX7.vcf.gz'
vcf = VCF(vcf_path)
print(f'Samples in VCF: {vcf.samples}')

# Peek at the first 3 variants
records = []
for i, variant in enumerate(vcf):
    records.append({
        'CHROM': variant.CHROM,
        'POS': variant.POS,
        'REF': variant.REF,
        'ALT': ",".join(variant.ALT),
        'QUAL': variant.QUAL,
        'FILTER': variant.FILTER,
        'GT': variant.gt_types[0] # 0=HOM_REF, 1=HET, 2=HOM_UNKNOWN, 3=HOM_ALT
    })

display(pd.DataFrame(records))

Samples in VCF: ['WGS_EX2312012']


,CHROM,POS,REF,ALT,QUAL,FILTER,GT
0,1,13273,G,C,385.769989,MQ40,3
1,1,13417,C,CGAGA,320.750000,MQ40,3
2,1,14248,T,G,59.770000,MQ40,1
3,1,14354,C,A,11.120000,LowQual;MQ40;QD2,1
4,1,14590,G,A,76.769997,MQ40,1
...,...,...,...,...,...,...,...
5012199,Un_JTFH01001998v1_decoy,1421,G,A,1487.770020,NaN,3
5012200,Un_JTFH01001998v1_decoy,1546,T,C,846.770020,NaN,3
5012201,Un_JTFH01001998v1_decoy,1613,A,T,376.769989,NaN,1
5012202,Un_JTFH01001998v1_decoy,1626,T,A,1101.770020,NaN,3


In [4]:
df = pd.DataFrame(records)

In [5]:
df.CHROM.value_counts()

CHROM
2                          388761
1                          383428
4                          339607
3                          321840
6                          303945
                            ...  
Un_JTFH01001913v1_decoy         1
Un_JTFH01001930v1_decoy         1
Un_JTFH01001933v1_decoy         1
Un_JTFH01001967v1_decoy         1
Un_JTFH01001976v1_decoy         1
Name: count, Length: 1690, dtype: int64

## Raw Genomic Data (.fastq.gz)
These are the massive raw sequencing files (~10GB each). We won't load them into memory, but we can use `gzip` to peek at the first 4 lines of one file.

In a FASTQ file, exactly **4 lines represent 1 single sequencing read**:
1. A sequence identifier with information about the sequencing run.
2. The sequence itself (the A, C, T, G, and N characters).
3. A separator line (starts with `+`).
4. The base quality scores (ASCII characters representing how confident the machine is in each letter).

In [6]:
import gzip
import os

fastq_path = '../data/WGS_EX2312012_HGWCNDSX7_S16_L001_R1_001.fastq.gz'
print(f'Reading the first 4 lines (1 read) of {os.path.basename(fastq_path)}:\n')
with gzip.open(fastq_path, 'rt') as f:
    for i in range(4):
        print(f.readline().strip())

Reading the first 4 lines (1 read) of WGS_EX2312012_HGWCNDSX7_S16_L001_R1_001.fastq.gz:

@A01973:164:HGWCNDSX7:1:1101:11957:1000 1:N:0:GTGACGGAGC+NGGACCGCCA
ANGATAGATAGCAAAACATTGCAGAACGATCAAAAAGTCAGAGATAGGAAGCGGGACGAGGAAAGGAACAGGCATGGAGGTCAACCCCAGAGTCTTAACATGCATCAGATGCATGTTCTGGTGGGCTGAGGCCAAGAAACATTGAAAAG
+
F#:FFFFFFFFFFFFFFFFF:FFFFFFFFF:FFFFFFFF,F:FFFFF::FFFFFFFFFFFFFFFFFF:FFFF:FFFFFFFFFFFFFF:,FFFFF,FF:FFFFFFFFFFF:FFF,FFFFFFFFFFFFFFFF:FFFFFFFFFFFFFFFF:F
